### Data Acquisition

Fetched data from CRSP via WRDS database and restructure in into point-in-time S&P500 daily panel data.

*   Trainning period: 2020-2023
*   Validation period: 2024
*  Test period: 2025



In [51]:
import wrds
db = wrds.Connection(wrds_username = 'sarahliuuuuuuuu')

Enter your WRDS username [sarahliuuuuuuuu]:sarahliuuuuuuuu
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: n
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [52]:
import numpy as np
import pandas as pd

In [53]:
START = "2019-01-01"
END = "2025-12-31"
VAL_START = "2024-01-01"
TEST_START = "2025-01-01"

In [54]:
MEMBER_TABLE = "dsp500list_v2"
DAILY_TABLE = "dsf_v2"
print(db.describe_table(library= "crsp", table = MEMBER_TABLE))
print(db.describe_table(library="crsp", table= DAILY_TABLE))

Approximately 2084 rows in crsp.dsp500list_v2.
         name  nullable         type                                   comment
0      permno      True      INTEGER                                    PERMNO
1       indno      True      INTEGER                                     INDNO
2  mbrstartdt      True         DATE         Constituent Membership Start Date
3    mbrenddt      True         DATE           Constituent Membership End Date
4      mbrflg      True  VARCHAR(12)               Constituent Membership Flag
5      indfam      True      INTEGER  INDFAM - CRSP Unique Index Family Number
Approximately 110257376 rows in crsp.dsf_v2.
                name  nullable             type  \
0             permno      True          INTEGER   
1           hdrcusip      True       VARCHAR(8)   
2             permco      True          INTEGER   
3              siccd      True          INTEGER   
4         nasdissuno      True          INTEGER   
5           yyyymmdd      True          INTEGER  

In [55]:
#S&P500 membership constituent list
START_M = 'mbrstartdt'
END_M = "mbrenddt"
members = db.raw_sql(f"""
SELECT permno, {START_M} as start, {END_M} as ending
FROM crsp.{MEMBER_TABLE}
""", date_cols= ["start","ending"])

members["ending"] = members["ending"].fillna(pd.Timestamp(END))
members = members[(members["ending"] >= START) & (members["ending"] <= END)]
permno_list = ",".join((str(int(p))) for p in members["permno"].unique())
print(members["permno"].nunique())

637


In [64]:
#Daily table with each stock's calendar day, return, price, volumn
D_DATE, D_RET, D_PRC, D_VOL = "dlycaldt", "dlyret", "dlyprc", "dlyvol"
chunck = []
for year in range(2019, 2026):
    dstock = db.raw_sql(f""" SELECT permno, {D_DATE} as date, {D_RET} as return, {D_PRC} as price, {D_VOL} as volumn
            FROM crsp.{DAILY_TABLE}
            WHERE permno IN ({permno_list})
            AND {D_DATE} BETWEEN '{year}-01-01' AND '{year}-12-31'
        """, date_cols = ["date"]
    )
    chunck.append(dstock)


In [65]:
panel_dt = pd.concat(chunck, ignore_index= True)
panel_dt.head()

,permno,date,return,price,volumn
0,10104,2019-01-02,0.00155,45.22,14320441.0
1,10104,2019-01-03,-0.00973,44.78,19868713.0
2,10104,2019-01-04,0.0431,46.71,20983953.0
3,10104,2019-01-07,0.015842,47.45,17967907.0
4,10104,2019-01-08,0.009062,47.88,16255688.0


In [66]:
last_row_permno = panel_dt.sort_values("date").groupby("permno").tail(1)
delisted = last_row_permno[last_row_permno["date"] < pd.Timestamp(END) - pd.Timedelta(days = 10)]
print("Delisted during researching period:",len(delisted))
print("The return on final trading day is empty:",delisted["return"].isna().sum())

Delisted during researching period: 59
The return on final trading day is empty: 2


In [67]:
#Flag "is in S&P500 now" for each row
flag = panel_dt[["permno","date"]].merge(members, on= "permno")
flag = flag[(flag["date"]> flag["start"]) & (flag["date"] < flag["ending"])]
flag = flag[["permno","date"]].drop_duplicates()
flag["isin_sp500"] = True
panel_dt = panel_dt.merge(flag, on=["permno","date"], how='left')
panel_dt["isin_sp500"] = panel_dt["isin_sp500"].fillna(False).astype(bool)

/tmp/ipykernel_545/1910034973.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  panel_dt["isin_sp500"] = panel_dt["isin_sp500"].fillna(False).astype(bool)


In [68]:
#train, test split, save the file
panel_dt["log_return"] = np.log1p(panel_dt["return"])
panel_dt.sort_values("date").reset_index(drop = True)

panel_dt["split"] = np.select([panel_dt["date"]< "2020-01-01", panel_dt["date"] < VAL_START, panel_dt["date"] < TEST_START],["warmup","train","val"],default= 'test')
print(panel_dt.head())


   permno       date    return  price      volumn  isin_sp500  log_return  \
0   10104 2019-01-02   0.00155  45.22  14320441.0        True    0.001549   
1   10104 2019-01-03  -0.00973  44.78  19868713.0        True   -0.009778   
2   10104 2019-01-04    0.0431  46.71  20983953.0        True    0.042197   
3   10104 2019-01-07  0.015842  47.45  17967907.0        True    0.015718   
4   10104 2019-01-08  0.009062  47.88  16255688.0        True    0.009021   

    split  
0  warmup  
1  warmup  
2  warmup  
3  warmup  
4  warmup  


In [71]:
import os

SAVE_DIR = "/content/drive/MyDrive/sp500_vol"
os.makedirs(SAVE_DIR, exist_ok=True)

panel_dt.to_parquet(f"{SAVE_DIR}/sp500_panel.parquet")